In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q transformers
!pip install -q accelerate
!pip install -q sentence-transformers
!pip install -q chromadb
!pip install -q sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 re

In [3]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Create mock data for multiple tables
patients_data = {
    'patient_id': [1, 2, 3, 4],
    'patient_name': ['Alex', 'Sarah', 'David', 'Emma'],
    'gender': ['Male', 'Female', 'Male', 'Female'],
    'age': [45, 60, 55, 30]
}

doctors_data = {
    'doctor_id': [101, 102],
    'doctor_name': ['Dr. Smith', 'Dr. Adams'],
    'specialty': ['Cardiology', 'Oncology']
}

appointments_data = {
    'appointment_id': [1001, 1002, 1003, 1004],
    'patient_id': [1, 2, 3, 1],
    'doctor_id': [101, 102, 101, 102],
    'appointment_date': ['2025-01-15', '2025-02-20', '2025-03-05', '2025-04-10']
}

# 2. Create SQLite DB Engine
engine = create_engine("sqlite:///hospital.db")

# 3. Store tables in the database
pd.DataFrame(patients_data).to_sql("patients", engine, index=False, if_exists='replace')
pd.DataFrame(doctors_data).to_sql("doctors", engine, index=False, if_exists='replace')
pd.DataFrame(appointments_data).to_sql("appointments", engine, index=False, if_exists='replace')

print("Database Created with tables: patients, doctors, appointments")

Database Created with tables: patients, doctors, appointments


In [4]:
query = "SELECT * FROM patients"

result = pd.read_sql(query, engine)

print(result)

   patient_id patient_name  gender  age
0           1         Alex    Male   45
1           2        Sarah  Female   60
2           3        David    Male   55
3           4         Emma  Female   30


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "defog/sqlcoder-7b-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model Loaded")

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model Loaded


In [6]:
schema = """
Table: patients
Columns:
- patient_id (INTEGER PRIMARY KEY)
- patient_name (VARCHAR)
- gender (VARCHAR)
- age (INTEGER)

Table: doctors
Columns:
- doctor_id (INTEGER PRIMARY KEY)
- doctor_name (VARCHAR)
- specialty (VARCHAR)

Table: appointments
Columns:
- appointment_id (INTEGER PRIMARY KEY)
- patient_id (INTEGER) -- Foreign Key to patients.patient_id
- doctor_id (INTEGER) -- Foreign Key to doctors.doctor_id
- appointment_date (DATE)
"""

In [7]:
# This question requires joining patients -> appointments -> doctors
question = "What are the names of the patients who have appointments with a doctor specializing in Cardiology? Also return the appointment date."

In [8]:
prompt = f"""
You are an expert medical SQLite SQL generator.

Rules:
- Return ONLY SQL
- Use SQLite syntax
- Use only provided schema
- No explanations

Schema:
{schema}

Question:
{question}

SQL:
"""

In [9]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are an expert medical SQLite SQL generator.

Rules:
- Return ONLY SQL
- Use SQLite syntax
- Use only provided schema
- No explanations

Schema:

Table: patients
Columns:
- patient_id (INTEGER PRIMARY KEY)
- patient_name (VARCHAR)
- gender (VARCHAR)
- age (INTEGER)

Table: doctors
Columns:
- doctor_id (INTEGER PRIMARY KEY)
- doctor_name (VARCHAR)
- specialty (VARCHAR)

Table: appointments
Columns:
- appointment_id (INTEGER PRIMARY KEY)
- patient_id (INTEGER) -- Foreign Key to patients.patient_id
- doctor_id (INTEGER) -- Foreign Key to doctors.doctor_id
- appointment_date (DATE)


Question:
What are the names of the patients who have appointments with a doctor specializing in Cardiology? Also return the appointment date.

SQL:
 SELECT p.patient_name, a.appointment_date FROM patients p JOIN appointments a ON p.patient_id = a.patient_id JOIN doctors d ON a.doctor_id = d.doctor_id WHERE d.specialty = 'Cardiology';


In [11]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Query Execution Result ---")
try:
    # Extract ONLY the text that comes after "SQL:" 
    # and clean up any markdown backticks
    clean_sql = response.split("SQL:")[-1].replace("```sql", "").replace("```", "").replace(";", "").strip()
    
    print(f"Executing Query:\n{clean_sql}\n")
    
    # Execute the generated SQL query using pandas
    result_df = pd.read_sql(clean_sql, engine)
    print("--- Output Data ---")
    print(result_df)
except Exception as e:
    print("Error executing query:", e)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- Query Execution Result ---
Executing Query:
SELECT p.patient_name, a.appointment_date FROM patients p JOIN appointments a ON p.patient_id = a.patient_id JOIN doctors d ON a.doctor_id = d.doctor_id WHERE d.specialty = 'Cardiology'

--- Output Data ---
  patient_name appointment_date
0         Alex       2025-01-15
1        David       2025-03-05
